# Netflix Catalog Analysis in Python

Reproduces the key findings from [Project 1](../project-01-netflix-content-analysis/README.md) with pandas, as an independent check of the Excel, MySQL and Power BI work.

**Question:** Which audiences does the catalog serve most heavily, and which genres define each?

Rating groups: Adult = TV-MA/R/NC-17 · Teen = TV-14/PG-13 · Older kids/Family = TV-PG/PG/TV-Y7/TV-Y7-FV · All kids = TV-Y/TV-G/G.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("../project-01-netflix-content-analysis/data/cleaned")
titles = pd.read_csv(DATA / "netflix_titles_clean.csv", na_values=["NULL"])
genres_wide = pd.read_csv(DATA / "listed_in.csv", na_values=["NULL"])
countries_wide = pd.read_csv(DATA / "countries.csv", na_values=["NULL"])
print(titles.shape)
titles.head()

## 1. Data checks

In [ ]:
assert titles["show_id"].is_unique
titles.isna().sum()

## 2. Movies vs. TV shows

In [ ]:
type_share = titles["type"].value_counts()
print(type_share)
print((type_share / len(titles) * 100).round(1))

## 3. Titles added per year

In [ ]:
titles["date_added"] = pd.to_datetime(titles["date_added"].str.strip(), format="%B %d, %Y", errors="coerce")
titles["year_added"] = titles["date_added"].dt.year
per_year = titles.groupby(["year_added", "type"]).size().unstack(fill_value=0)
per_year.plot(kind="bar", stacked=True, figsize=(9, 4), title="Titles added per year (2021 is partial)")
plt.ylabel("Titles")
plt.tight_layout()
per_year.tail(4)

## 4. Audience share by rating group

In [ ]:
GROUPS = {
    "Adult": ["TV-MA", "R", "NC-17"],
    "Teen": ["TV-14", "PG-13"],
    "Older kids/Family": ["TV-PG", "PG", "TV-Y7", "TV-Y7-FV"],
    "All kids": ["TV-Y", "TV-G", "G"],
}
rating_to_group = {r: g for g, rs in GROUPS.items() for r in rs}
titles["segment"] = titles["rating"].map(rating_to_group)

print("Titles with no valid rating group:", titles["segment"].isna().sum())
seg = titles["segment"].value_counts()
seg_share = (seg / seg.sum() * 100).round(1)
pd.DataFrame({"titles": seg, "share_%": seg_share})

In [ ]:
seg_share.sort_values().plot(kind="barh", title="Share of catalog by audience segment (%)")
plt.tight_layout()

## 5. Top genres overall and within each segment

In [ ]:
genres = (genres_wide.melt(id_vars="show_id", value_name="genre")
          .dropna(subset=["genre"])[["show_id", "genre"]]
          .drop_duplicates())
print(len(genres), "title-genre rows,", genres["genre"].nunique(), "genres")
genres["genre"].value_counts().head(10)

In [ ]:
g = genres.merge(titles[["show_id", "segment"]], on="show_id").dropna(subset=["segment"])
seg_size = titles["segment"].value_counts()
top = (g.groupby(["segment", "genre"]).size().rename("n").reset_index())
top["share_of_segment_%"] = (top["n"] / top["segment"].map(seg_size) * 100).round(0)
top = top.sort_values(["segment", "n"], ascending=[True, False]).groupby("segment").head(4)
top

## 6. Countries

In [ ]:
countries = (countries_wide.melt(id_vars="show_id", value_name="country")
             .dropna(subset=["country"])[["show_id", "country"]]
             .drop_duplicates())
countries["country"] = countries["country"].str.strip()
countries = countries[countries["country"] != ""]
print(countries["country"].nunique(), "countries")
countries["country"].value_counts().head(5)

## Takeaways

Compare the figures above with the Project 1 README: the segment shares, top genres and country counts should match the SQL and Power BI results. As in the original analysis, the catalog describes what is *offered*, not what people *watch*, so these are hypotheses to test against viewership data.